<!-- codex_annotation: script_overview -->
# 基础模板匹配坐标生成初版

step1 copy 的早期版本：读取完整图和 tile，直接全局模板匹配并输出坐标文件。

注释说明：
- 当前 OUTPUT_FILE 后缀写成 .tif，但内容实际是 TileConfiguration 文本，建议使用 .txt。


In [1]:
import os
import cv2
import tifffile as tiff
import numpy as np


In [2]:
#==========================
# 路径设置
# ==========================

FULL_IMAGE = r"D:\01.analysis\11.test\DAPI\fullbrain.tif"

TILE_DIR = r"D:\01.analysis\11.test\DAPI\tiles"

OUTPUT_FILE = r"D:\01.analysis\11.test\DAPI\TileConfiguration.tif"


In [3]:
# ==========================
# 读取完整脑片
# ==========================

print("Loading full brain...")

full = tiff.imread(FULL_IMAGE)

if full.ndim > 2:
    full = full[0]

full = cv2.normalize(
    full,
    None,
    0,
    255,
    cv2.NORM_MINMAX
).astype(np.uint8)

print("Full brain size:", full.shape)

Loading full brain...
Full brain size: (11952, 12832)


In [4]:
# ==========================
# Tile列表
# ==========================

tile_files = sorted([
    f for f in os.listdir(TILE_DIR)
    if f.endswith(".tif")
])

results = []

In [5]:
# ==========================
# 搜索Tile位置
# ==========================

for tile_name in tile_files:

    print("\nSearching:", tile_name)

    tile_path = os.path.join(
        TILE_DIR,
        tile_name
    )

    tile = tiff.imread(tile_path)

    if tile.ndim > 2:
        tile = tile[0]

    tile = cv2.normalize(
        tile,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    print(
        "Tile size:",
        tile.shape
    )

    result = cv2.matchTemplate(
        full,
        tile,
        cv2.TM_CCOEFF_NORMED
    )

    _, max_val, _, max_loc = cv2.minMaxLoc(result)

    x, y = max_loc

    print(
        f"Position = ({x},{y}) "
        f"Score={max_val:.3f}"
    )

    results.append(
        (
            tile_name,
            x,
            y
        )
    )



Searching: tile_01.tif
Tile size: (6431, 2896)
Position = (919,87) Score=0.532

Searching: tile_02.tif
Tile size: (6443, 3786)
Position = (3329,178) Score=0.486

Searching: tile_03.tif
Tile size: (6487, 6486)
Position = (6071,245) Score=0.497

Searching: tile_04.tif
Tile size: (6484, 6483)
Position = (561,5468) Score=0.475

Searching: tile_05.tif
Tile size: (6455, 4689)
Position = (4541,5497) Score=0.425

Searching: tile_06.tif
Tile size: (6507, 4706)
Position = (7997,5445) Score=0.384


In [6]:
# ==========================
# 输出Fiji坐标文件
# ==========================

with open(
    OUTPUT_FILE,
    "w"
) as f:

    f.write("dim = 2\n\n")

    for tile_name,x,y in results:

        f.write(
            f"{tile_name}; ; ({x},{y})\n"
        )

print("\nSaved:")
print(OUTPUT_FILE)


Saved:
D:\01.analysis\11.test\DAPI\TileConfiguration.tif
